# IOAI — 2026 Contest Molecular Energy Dipole (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.listdir('data'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-contest-molecular-energy-dipole/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('데이터 준비:', sorted(os.listdir('data'))[:8])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Molecular — 모범답안 (물리 불변 특징 + 랜덤포레스트)

**에너지**는 원자 조성에 거의 가법적이므로 원자 개수 + 거리행렬 기반 불변 특징(Coulomb 합, 밀도, 최소/평균
거리)으로 잘 예측된다. **쌍극자**는 기하에 의존하므로 전하가중 중심변위 등 기하 특징을 추가한다. 각 타깃을
랜덤포레스트로 회귀. 점수 ≈ 0.82 (참고점수 0.88).

In [ ]:
import pandas as pd, numpy as np, glob, os
ref = pd.read_csv("data/train/training_ref.csv")           # molecule #, energy, dipole moment
ref.columns = ["mol", "energy", "dipole"]
test_ids = pd.read_csv("data/test/test_list.csv")["molecule #"].tolist()
PROP = {"H":(1,2.20),"C":(6,2.55),"N":(7,3.04),"O":(8,3.44),"F":(9,3.98)}   # (nuclear charge, electronegativity)
ATOMS = "HCNOF"

def parse_xyz(path):
    ls = open(path).read().splitlines(); n = int(ls[0])
    syms, xyz = [], []
    for l in ls[2:2+n]:
        p = l.split(); syms.append(p[0]); xyz.append([float(v) for v in p[1:4]])
    return syms, np.array(xyz)
print("train molecules", len(ref), "| test molecules", len(test_ids))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def features(syms, xyz):
    cnt = [syms.count(a) for a in ATOMS]
    Z = np.array([PROP[s][0] for s in syms], float)
    D = np.sqrt(((xyz[:,None]-xyz[None])**2).sum(-1)); np.fill_diagonal(D, np.inf)
    coul = Z[:,None]*Z[None]/D; coul[~np.isfinite(coul)] = 0
    inv = 1/D; inv[~np.isfinite(inv)] = 0
    c = xyz - xyz.mean(0)
    dip = [np.linalg.norm((c*Z[:,None]).sum(0)), np.abs(c).sum()]
    return cnt + [len(syms), coul.sum(), inv.sum(),
                  D[np.isfinite(D)].min() if np.isfinite(D).any() else 0.0] + dip

def build(ids, base):
    F = []
    for m in ids:
        s, xyz = parse_xyz(f"{base}/molecule_{m}/coordinate.xyz"); F.append(features(s, xyz))
    return np.array(F)

Xtr = build(ref.mol, "data/train")
Xte = build(test_ids, "data/test")
er = RandomForestRegressor(n_estimators=400, random_state=0).fit(Xtr, ref.energy)
dr = RandomForestRegressor(n_estimators=400, random_state=0).fit(Xtr, ref.dipole)
ep, dp = er.predict(Xte), dr.predict(Xte)
pd.DataFrame({"molecule #": test_ids, "energy": ep, "dipole moment": dp}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(test_ids))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)